In [2]:
import pandas as pd

In [ ]:
thoracic_tabsyn = pd.read_csv('synthetic/thoracic_surgery/tabsyn.csv')
thoracic_ganos = pd.read_csv('synthetic/thoracic_surgery/ganos.csv')
thoracic_codi = pd.read_csv('synthetic/thoracic_surgery/codi.csv')
thoracic_findiff = pd.read_csv('synthetic/thoracic_surgery/findiff.csv')
thoracic_great = pd.read_csv('synthetic/thoracic_surgery/great.csv')
thoracic_ctgan = pd.read_csv('synthetic/thoracic_surgery/ctgan.csv')

models = ["tabsyn", "ganos", "codi", "ctgan"]

# original data

thoracic = pd.read_csv('data/thoracic_surgery/train.csv')

In [7]:
thoracic

,PRE4,PRE5,PRE6,PRE7,PRE8,PRE9,PRE10,PRE11,PRE14,PRE17,...,PRE30,PRE32,AGE,DGN_DGN2,DGN_DGN3,DGN_DGN4,DGN_DGN5,DGN_DGN6,DGN_DGN8,Risk1Yr
0,2.04,1.80,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,64.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,3.24,2.40,1.0,1.0,1.0,0.0,0.0,0.0,3.0,0.0,...,1.0,0.0,55.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,2.40,1.24,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,62.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,2.40,1.64,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,64.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,3.20,2.52,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,75.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,3.68,3.20,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,55.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
406,3.24,1.64,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,63.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
407,5.00,4.04,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,60.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
408,2.58,1.64,2.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,63.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
thoracic_ganos.Risk1Yr.value_counts()

NameError: name 'thoracic_ganos' is not defined

In [18]:
data = pd.read_csv('/home/pcrespo/awesome-tab-augmentation/baselines/ganos/thoracic_surgery/synthetic_balanced.csv')

In [19]:
data['Risk1Yr'].value_counts()

Risk1Yr
0.0    349
1.0    349
Name: count, dtype: int64

In [ ]:
alpha_precissions = {}
beta_recalls = {}

thoracic_df = pd.DataFrame(columns=["Alpha Precission", "Beta Recall"])
for model in models:
    quality_path = "eval/quality/thoracic_surgery/" + model + ".txt"
    with open(quality_path, "r") as f:
        lines = f.readlines()
        
        alpha_precission = float(lines[0].split(":")[1].strip())
        beta_recall = float(lines[1].split(":")[1].strip())
        
        alpha_precissions[model] = alpha_precission
        beta_recalls[model] = beta_recall

for model in models:
    thoracic_df.loc[model] = [alpha_precissions[model], beta_recalls[model]]

thoracic_df

,Alpha Precission,Beta Recall
tabsyn,0.950765,0.539837
ganos,0.625854,0.018211
codi,0.610407,0.440976
ctgan,0.865069,0.122764


In [16]:
shape_scores = {}
trend_scores = {}

thoracic_df = pd.DataFrame(columns=["Shape Score", "Trend Score"])

for model in models:
    quality_path = "eval/density/thoracic_surgery/" + model + "/quality.txt"
    with open(quality_path, "r") as f:
        lines = f.readlines()
        
        alpha_precission = float(lines[0].split(":")[1].strip())
        beta_recall = float(lines[1].split(":")[1].strip())
        
        alpha_precissions[model] = alpha_precission
        beta_recalls[model] = beta_recall

for model in models:
    thoracic_df.loc[model] = [alpha_precissions[model], beta_recalls[model]]

thoracic_df

,Shape Score,Trend Score
tabsyn,0.971286,0.939129
ganos,0.827605,0.747612
codi,0.923614,0.818844
ctgan,0.917406,0.847262


In [38]:
def process_results(lines):
    """
    Process lines of text into a dictionary of metric results.

    Args:
        lines (list of str): Lines read from the file.

    Returns:
        dict: Dictionary with metrics as keys and their values.
    """
    results = {}
    current_key = None

    for line in lines:
        line = line.strip()
        if line.endswith(":"):  # Detect metric name
            current_key = line[:-1]
            results[current_key] = ""  # Initialize the value for this key
        elif current_key:  # Accumulate values for the current key
            results[current_key] += line + " "

    # Parse the accumulated values into Python objects if possible
    for key, value in results.items():
        try:
            results[key] = eval(value.strip())  # Safely parse the value
        except:
            results[key] = value.strip()  # Keep as string if parsing fails

    return results

In [43]:
metrics_all = {}

for model in models:
    quality_path = "eval/metrics/thoracic_surgery/" + model + "/metrics.txt"
    with open(quality_path, "r") as f:
        lines = f.readlines()
        results = process_results(lines)

        metrics_all[model] = results

In [50]:
df_metrics = pd.DataFrame.from_dict(metrics_all, orient="index")


In [51]:
df_metrics

,Kolmogorov-Smirnov Test,Total Variation Distance,Kullback-Leibler Divergence,Hellinger Distance,Mean Absolute Error Probability,Pairwise Correlation Difference
tabsyn,"{2: {'statistic': 0.041463414634146344, 'p_val...","{2: 0.05609756097560973, 3: 0.0097560975609755...","{2: 0.007877651011388006, 3: 0.000832840750850...","{2: 0.04435995323462588, 3: 0.0144258618956954...","{2: 0.11219512195121946, 3: 0.0195121951219511...",0 0.067310 1 0.035596 14 0.034324 d...
ganos,"{2: {'statistic': 0.04390243902439024, 'p_valu...","{2: 0.043902439024390255, 3: 0.034146341463414...","{2: 0.0153229907556629, 3: 0.00776784102653821...","{2: 0.06174118358371346, 3: 0.0439943372794397...","{2: 0.08780487804878051, 3: 0.0682926829268293...",0 0.394327 1 0.532106 14 0.504032 d...
codi,"{2: {'statistic': 0.11707317073170732, 'p_valu...","{2: 0.11707317073170734, 3: 0.0634146341463414...","{2: 0.5888322861709189, 3: 0.02361415930325461...","{2: 0.1843391870518889, 3: 0.07650742455866216...","{2: 0.23414634146341468, 3: 0.1268292682926829...",0 0.172132 1 0.193851 14 0.202433 d...
ctgan,"{2: {'statistic': 0.05853658536585366, 'p_valu...","{2: 0.05853658536585363, 3: 0.0414634146341463...","{2: 0.011073985346502483, 3: 0.011065640491200...","{2: 0.05258321604900408, 3: 0.0524776025707720...","{2: 0.11707317073170725, 3: 0.0829268292682927...",0 1.187538 1 1.090680 14 0.549633 d...
